# Fine-tuning Example: GPT-2 for Custom Text Generation

This notebook demonstrates how to fine-tune a pre-trained GPT-2 model on custom text data using the Small Language Model framework.

## Table of Contents
1. Setup and Installation
2. Load Pre-trained Model
3. Prepare Training Data
4. Fine-tune the Model
5. Generate Text
6. Evaluate the Model

## 1. Setup and Installation

In [ ]:
# Install required packages
# !pip install tensorflow transformers datasets

import os
import sys
sys.path.insert(0, os.path.dirname(os.getcwd()))

import tensorflow as tf
print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 2. Load Pre-trained Model

In [ ]:
from src.model.pretrained_wrapper import PretrainedModelWrapper

# Load GPT-2 for text generation
model = PretrainedModelWrapper(
    model_name='gpt2',
    task='text_generation',
)
tokenizer = model.tokenizer
tokenizer.pad_token = tokenizer.eos_token

param_info = model.count_parameters()
print(f"Total parameters: {param_info['total']:,}")
print(f"Trainable parameters: {param_info['trainable']:,}")

## 3. Prepare Training Data

In [ ]:
from src.data.preprocessing import DataPreprocessor

# Sample training texts (replace with your actual data)
sample_texts = [
    "The quick brown fox jumps over the lazy dog.",
    "Machine learning is transforming the world.",
    "Once upon a time, in a kingdom far away.",
    "The scientist discovered a remarkable phenomenon.",
    "In the age of artificial intelligence, creativity flourishes.",
] * 20  # Repeat to create more training data

preprocessor = DataPreprocessor(tokenizer_name='gpt2', max_seq_length=512)
cleaned_texts = preprocessor.clean_texts(sample_texts)

# Split into train/val
train_texts, val_texts, test_texts = preprocessor.split_dataset(
    cleaned_texts, ratios=(0.8, 0.1, 0.1)
)

print(f"Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}")

# Save splits to disk
os.makedirs('/tmp/finetuning_data', exist_ok=True)
preprocessor.save_splits(train_texts, val_texts, test_texts, '/tmp/finetuning_data')

In [ ]:
from src.data.dataset import TextDataset

# Create datasets
train_dataset = TextDataset(
    file_path='/tmp/finetuning_data/train.txt',
    tokenizer=tokenizer,
    max_seq_length=512,
)
val_dataset = TextDataset(
    file_path='/tmp/finetuning_data/val.txt',
    tokenizer=tokenizer,
    max_seq_length=512,
)

train_ds = train_dataset.get_tf_dataset(batch_size=4, shuffle=True)
val_ds = val_dataset.get_tf_dataset(batch_size=4, shuffle=False)

print(f"Training examples: {len(train_dataset)}")
print(f"Validation examples: {len(val_dataset)}")

## 4. Fine-tune the Model

In [ ]:
from src.training.trainer import Trainer

optimizer = tf.keras.optimizers.AdamW(
    learning_rate=5e-5,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    optimizer=optimizer,
    train_dataset=train_ds,
    val_dataset=val_ds,
    num_epochs=3,
    gradient_accumulation_steps=4,
    output_dir='/tmp/finetuned_gpt2',
    logging_steps=10,
    save_steps=100,
    early_stopping_patience=3,
)

print("Starting fine-tuning...")
history = trainer.train()
print("\nFine-tuning complete!")
print(f"Final train loss: {history['train_loss'][-1]:.4f}" if history['train_loss'] else "No loss recorded")

## 5. Generate Text

In [ ]:
from src.inference.predictor import Predictor

predictor = Predictor(
    model=model,
    tokenizer=tokenizer,
    task='text_generation',
)

prompts = [
    "The future of machine learning",
    "Once upon a time",
    "In the field of artificial intelligence",
]

print("Generated Text Samples")
print("=" * 60)
for prompt in prompts:
    result = predictor.generate(
        prompt,
        max_new_tokens=100,
        temperature=0.8,
        top_k=50,
        top_p=0.9,
    )
    print(f"\nPrompt: {prompt}")
    print(f"Generated: {result['generated_text'][0]}")

## 6. Plot Training Metrics

In [ ]:
import matplotlib.pyplot as plt

if history.get('train_loss'):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history['train_loss'], label='Train Loss', color='blue')
    if history.get('val_loss'):
        axes[0].plot(history['val_loss'], label='Val Loss', color='orange')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training and Validation Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    if history.get('val_perplexity'):
        axes[1].plot(history['val_perplexity'], label='Val Perplexity', color='green')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Perplexity')
        axes[1].set_title('Validation Perplexity')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('/tmp/training_curves.png', dpi=150)
    plt.show()
    print("Training curves saved to /tmp/training_curves.png")
else:
    print("No training history to plot.")